# AI for Health — PGE5 Day 1

## Task 1: Health Data Modalities — Deep Python Library Exploration
## Task 2: DICOM Metadata Explorer

This notebook covers two things:
1. **Task 1** — a deep exploration of Python libraries across the four main health-data modalities, with **runnable examples on real data** for every modality. Where multiple libraries solve the same problem, we benchmark and compare them.
2. **Task 2** — the assignment's DICOM Metadata Explorer, exactly per the spec, run against **SIIM_SMALL** (250 real chest-X-ray DICOMs, ~30 MB, downloaded automatically).

### What's in Task 1

| § | Modality | What we actually do |
|---|---|---|
| 1.1 | X-ray (DICOM) | `pydicom` vs `SimpleITK` vs `imageio` — same file, three libraries, benchmarked |
| 1.2 | MRI (NIfTI) | Real 4D MRI volume — voxel spacing, affine matrix, axial slicing, time series |
| 1.3 | Whole-Slide Imaging | OpenSlide tile model, pyramid levels, region reads, RGBA, MPP property |
| 1.4 | EHR / clinical text | `medkit` regex matching + **negation detection** + sentence segmentation pipeline |
| 1.5 | Cross-modality summary | Trade-offs, when to pick which library |

### What's in Task 2
Strictly per the assignment spec: load all DICOMs from a folder, extract **modality**, **patient age**, **acquisition date**, **body part examined**, store in a `pandas` DataFrame, generate summary statistics.


---
# Task 1 — Health Data Modalities

## 1.0 One-time install

Run this cell once per environment.

In [ ]:
# --- One-shot install for Colab. Safe to re-run; --quiet keeps output clean. ---
# Core imaging
!pip install pydicom nibabel SimpleITK imageio --quiet
# Helper for downloading the SIIM_SMALL dataset in Task 2
!pip install fastai --quiet
# EHR / clinical NLP — note: PyPI package is 'medkit-lib', imported as 'medkit'
!pip install medkit-lib --quiet
# WSI (the Python wrapper alone — needs the OpenSlide C library too).
# Uncomment the two lines below if you actually want to run the WSI cells.
# !apt-get install -y openslide-tools >/dev/null 2>&1
# !pip install openslide-python --quiet

import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
print("Core libs imported.")

## 1.1 X-ray — `pydicom` vs `SimpleITK` vs `imageio`

X-rays are **DICOM** files: a binary container holding a pixel array plus a metadata header (patient, device, acquisition parameters).

Three Python libraries can read a DICOM, but they're designed for different purposes:

| Library | Strength | Trade-off |
|---|---|---|
| **`pydicom`** | Full access to every tag; pure Python | No image processing — pixels are a numpy array, that's it |
| **`SimpleITK`** | Exposes spatial metadata (spacing, origin, direction) as first-class fields; works with ITK image-processing pipeline | Heavier import; opinionated about geometry |
| **`imageio`** | Same API as for PNG/JPG — drop-in | Loses most of the DICOM header; spotty support for compressed transfer syntaxes |

Below we read the **same** file with all three and compare what each one gives us.


In [ ]:
# A synthetic chest-X-ray-style DICOM so the comparison is reproducible
# (the SIIM_SMALL data used later in Task 2 has compressed JPEG pixel data
# that imageio's reader trips over; using a vanilla uncompressed file here
# isolates the library-API differences from transfer-syntax issues).
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import generate_uid, ExplicitVRLittleEndian
import pydicom

os.makedirs("compare", exist_ok=True)
suid = generate_uid()
fm = FileMetaDataset()
fm.MediaStorageSOPClassUID    = "1.2.840.10008.5.1.4.1.1.2"
fm.MediaStorageSOPInstanceUID = suid
fm.TransferSyntaxUID          = ExplicitVRLittleEndian

ds = FileDataset("compare/demo.dcm", {}, file_meta=fm, preamble=b"\0"*128)
ds.PatientID, ds.PatientName     = "DEMO001", "Demo^Subject"
ds.PatientAge, ds.PatientSex     = "045Y", "M"
ds.Modality, ds.BodyPartExamined = "CR", "CHEST"
ds.StudyDate, ds.AcquisitionDate = "20240101", "20240101"
ds.Manufacturer                  = "DemoCorp"
ds.StudyInstanceUID  = generate_uid()
ds.SeriesInstanceUID = generate_uid()
ds.SOPInstanceUID    = suid
ds.SOPClassUID       = fm.MediaStorageSOPClassUID
ds.PixelSpacing      = [0.143, 0.143]   # mm/pixel — real-world value for CR

arr = (np.random.rand(512, 512) * 4095).astype(np.uint16)
ds.Rows, ds.Columns = arr.shape
ds.BitsAllocated, ds.BitsStored, ds.HighBit = 16, 12, 11
ds.PixelRepresentation, ds.SamplesPerPixel  = 0, 1
ds.PhotometricInterpretation = "MONOCHROME2"
ds.PixelData = arr.tobytes()
ds.save_as("compare/demo.dcm", enforce_file_format=True)

print("Test file: compare/demo.dcm (512x512 16-bit MONOCHROME2 CR)")

In [ ]:
# 1) pydicom
import pydicom

t0 = time.perf_counter()
ds_pyd = pydicom.dcmread("compare/demo.dcm")
arr_pyd = ds_pyd.pixel_array
t_pyd = (time.perf_counter() - t0) * 1000

print("--- pydicom ---")
print(f"  read time:        {t_pyd:.2f} ms")
print(f"  pixel array:      shape={arr_pyd.shape}, dtype={arr_pyd.dtype}")
print(f"  Modality:         {ds_pyd.Modality}")
print(f"  BodyPart:         {ds_pyd.BodyPartExamined}")
print(f"  PixelSpacing:     {ds_pyd.PixelSpacing}")
print(f"  total tags:       {len(ds_pyd)}")

In [ ]:
# 2) SimpleITK
import SimpleITK as sitk

t0 = time.perf_counter()
img_sitk = sitk.ReadImage("compare/demo.dcm")
arr_sitk = sitk.GetArrayFromImage(img_sitk)
t_sitk = (time.perf_counter() - t0) * 1000

print("--- SimpleITK ---")
print(f"  read time:        {t_sitk:.2f} ms")
print(f"  pixel array:      shape={arr_sitk.shape}, dtype={arr_sitk.dtype}")
print(f"  Modality (0008|0060):  {img_sitk.GetMetaData('0008|0060')}")
print(f"  Spacing (mm):     {img_sitk.GetSpacing()}")
print(f"  Origin:           {img_sitk.GetOrigin()}")
print(f"  Direction matrix: {img_sitk.GetDirection()}")
print(f"  metadata keys:    {len(img_sitk.GetMetaDataKeys())}")
print()
print("Note: SimpleITK promotes the 2D X-ray to a 3D image with size-1 z-axis")
print("because ITK is fundamentally an N-D imaging library. Use arr_sitk[0]")
print("to recover the 2D plane.")

In [ ]:
# 3) imageio
import imageio.v3 as iio

t0 = time.perf_counter()
try:
    arr_iio = iio.imread("compare/demo.dcm")
    t_iio = (time.perf_counter() - t0) * 1000
    print("--- imageio ---")
    print(f"  read time:        {t_iio:.2f} ms")
    print(f"  pixel array:      shape={arr_iio.shape}, dtype={arr_iio.dtype}")
    print(f"  metadata:         {iio.immeta('compare/demo.dcm')}")
except Exception as e:
    print("--- imageio ---")
    print(f"  FAILED: {type(e).__name__}: {e}")
    print()
    print("Real-world finding: imageio's DICOM plugin (a SimpleITK wrapper) can")
    print("trip on tags that pydicom and SimpleITK handle just fine. For DICOM")
    print("work, prefer pydicom or SimpleITK directly.")

**Takeaway.**

- Use **`pydicom`** when you need full header access (cohort selection, anonymization, tag inspection). It's the fastest and most complete for DICOM-specific work.
- Use **`SimpleITK`** when you need spatial geometry (spacing, origin, direction) — i.e. when the pixels are going to be resampled, registered, or fed to an ITK-based pipeline.
- **`imageio`** is convenient when you want to read DICOM as "just an image" alongside PNG/JPG, but expect occasional failures on real-world files; not the right tool when metadata or geometry matter.


## 1.2 MRI — `nibabel` on a real 4D NIfTI volume

MRI data lives mostly in **NIfTI** (`.nii` / `.nii.gz`) in research. NIfTI is a single-file 3D/4D volume with a header that carries:
- the voxel-to-world **affine matrix** (mm coordinates)
- voxel dimensions (`zooms`)
- axis orientation codes (e.g. `R, A, S` — Right, Anterior, Superior)

We use a small **real** 4D MRI sample shipped with `nibabel` — no download needed. It's a 128×96×24 volume with 2 time points (so 4D).

In [ ]:
import nibabel as nib

nib_test_data = os.path.join(os.path.dirname(nib.__file__), "tests/data")
sample_path = os.path.join(nib_test_data, "example4d.nii.gz")
print("Using NIfTI sample at:", sample_path)
print("File size:", os.path.getsize(sample_path), "bytes")

img = nib.load(sample_path)
print()
print("--- NIfTI geometry ---")
print(f"  shape:        {img.shape}             # (X, Y, Z, T)")
print(f"  data dtype:   {img.get_data_dtype()}")
print(f"  voxel sizes:  {tuple(round(float(z), 3) for z in img.header.get_zooms())}  (mm, mm, mm, ms)")
print(f"  units:        {img.header.get_xyzt_units()}")
print(f"  orientation:  {nib.aff2axcodes(img.affine)}")
print()
print("--- Affine matrix (voxel -> world mm) ---")
print(np.round(img.affine, 2))

In [ ]:
# Load the actual voxel data and look at intensity distribution
data = img.get_fdata()
print("Data array:", data.shape, data.dtype)
print(f"Intensity range: [{data.min():.0f}, {data.max():.0f}]")
print(f"Mean intensity:  {data.mean():.1f}")

# Visualise the three canonical anatomical planes from the first time point
vol0 = data[..., 0]
mid_x, mid_y, mid_z = [s // 2 for s in vol0.shape]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(vol0[mid_x, :, :].T, cmap="gray", origin="lower", aspect="auto")
axes[0].set_title(f"Sagittal slice (X={mid_x})")
axes[1].imshow(vol0[:, mid_y, :].T, cmap="gray", origin="lower", aspect="auto")
axes[1].set_title(f"Coronal slice (Y={mid_y})")
axes[2].imshow(vol0[:, :, mid_z].T, cmap="gray", origin="lower", aspect="auto")
axes[2].set_title(f"Axial slice (Z={mid_z})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 4D NIfTI also carries a temporal axis — show the two time points side by side
if data.shape[-1] >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    for t in range(2):
        axes[t].imshow(data[:, :, mid_z, t].T, cmap="gray", origin="lower", aspect="auto")
        axes[t].set_title(f"t = {t}")
        axes[t].axis("off")
    plt.tight_layout()
    plt.show()
    print(f"This volume has {data.shape[-1]} time points — typical of fMRI series.")

**Takeaway.** `nibabel` gives you a clean separation between:

- **data** (`img.get_fdata()`) — the voxel array,
- **header** (`img.header`) — the encoded format-specific tags,
- **affine** (`img.affine`) — the geometry that converts voxel indices to mm coordinates in the scanner frame.

That third item is the whole point of a medical-imaging library vs a generic image library. Two MRI volumes acquired in different orientations cannot be compared by stacking their pixel arrays — they have to be resampled through their affines first.

For deep-learning pipelines, **`MONAI`** wraps `nibabel` with transforms and data loaders tailored to medical images; for clinical tools, **`SimpleITK`** offers the same geometry treatment as for DICOM (Section 1.1).


## 1.3 Whole-Slide Imaging — `openslide-python`

Whole-slide pathology images are **gigapixel pyramidal TIFFs** (often 100,000 × 100,000 pixels, 1–4 GB each). You can't load them into memory; the library streams tiles from disk at the resolution you ask for.

The code below is the canonical workflow. Running it requires:
1. The OpenSlide C library installed at the system level (`apt install openslide-tools` on Linux, `brew install openslide` on macOS).
2. The `openslide-python` Python wrapper.
3. A `.svs` (or `.ndpi`, `.mrxs`, `.tiff`) slide file — download a small one from <https://openslide.cs.cmu.edu/download/openslide-testdata/Aperio/> (e.g. `CMU-1.svs`, ~170 MB).

```python
import openslide

slide = openslide.OpenSlide("CMU-1.svs")

# --- Geometry of the pyramid ---
print("Slide dimensions at level 0 (full res):", slide.dimensions)
# e.g. (46000, 32914) — 1.5 gigapixels

print("Number of pyramid levels:", slide.level_count)
print("Per-level dimensions:    ", slide.level_dimensions)
print("Per-level downsamples:   ", slide.level_downsamples)
# Each level downsamples by ~4x; level N is small enough to display whole

# --- Scanner metadata ---
print("MPP-x (microns/pixel):", slide.properties.get("openslide.mpp-x"))
print("MPP-y (microns/pixel):", slide.properties.get("openslide.mpp-y"))
print("Objective power:      ", slide.properties.get("openslide.objective-power"))
print("Vendor:               ", slide.properties.get("openslide.vendor"))

# --- Read a tile from full resolution ---
# (location is in level-0 coords even if level > 0; size is in pixels of the requested level)
tile = slide.read_region(location=(20000, 15000), level=0, size=(1024, 1024))
print("Tile type:", type(tile))    # PIL.Image, mode='RGBA'
tile_rgb = tile.convert("RGB")

# --- Or grab the entire slide as a thumbnail ---
thumbnail = slide.get_thumbnail(size=(1024, 1024))

# Show both
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(thumbnail)
axes[0].set_title(f"Thumbnail ({thumbnail.size})")
axes[0].axis("off")
axes[1].imshow(tile_rgb)
axes[1].set_title("1024x1024 tile @ level 0 (full res)")
axes[1].axis("off")
plt.show()
```

**Why this design matters.**

- A 100,000 × 100,000 RGB image at 8 bits per channel is 30 GB uncompressed. `OpenSlide` reads only the tiles you ask for — typical inference pipelines tile the slide at level 0 (e.g. 256×256 patches with overlap), process each tile with a CNN, and stitch the outputs.
- The **microns-per-pixel** (`openslide.mpp-x`, ~0.25 µm/pixel at 40× objective) is critical: deep-learning models trained on one scanner type often need rescaling to match another.
- The thumbnail uses the smallest pyramid level and is cheap to compute — use it for whole-slide visual QC.

For GPU-accelerated tile reading at scale (whole-cohort training), use **`cuCIM`** (NVIDIA RAPIDS) which is a drop-in OpenSlide replacement that decodes on the GPU. For pure-Python pipelines, **`tiffslide`** is a faster `OpenSlide`-compatible alternative for TIFF-based formats (Aperio SVS, Hamamatsu NDPI, Philips iSyntax not supported).


## 1.4 EHR / Clinical Notes — `medkit` pipeline with negation detection

Electronic Health Records are mostly **free text** — discharge summaries, progress notes, radiology reports. The task is clinical NLP: entity recognition (diseases, drugs, anatomy), **negation/uncertainty detection**, normalisation to ontologies (UMLS, SNOMED-CT, ICD-10).

A naïve regex `\bpneumonia\b` matches three very different sentences:
1. *"Patient has pneumonia"* — positive finding ✅
2. *"No evidence of pneumonia"* — negated, the patient does **not** have it ❌
3. *"Rule out pneumonia"* — uncertainty, neither confirmed nor denied ❓

Without negation detection, clinical NLP overstates disease prevalence dramatically. This is the whole reason a clinical-NLP framework like `medkit` exists vs. just calling `re.findall`.

> ⚠ Note: the medkit example in the original course notebook (`rules=[{"label": ..., "regexp": ...}, ...]`) was written against an older API. Current `medkit-lib` requires `RegexpMatcherRule` objects, used below.


In [ ]:
from medkit.core.text import TextDocument
from medkit.text.ner import RegexpMatcher, RegexpMatcherRule
from medkit.text.context import NegationDetector, NegationDetectorRule
from medkit.text.segmentation import SentenceTokenizer

# A more realistic clinical note: contains positive findings, a negation,
# a drug mention, and a vital sign.
note_text = (
    "67-year-old male with a history of type 2 diabetes and hypertension. "
    "Patient reports chest pain and shortness of breath since yesterday. "
    "Currently taking metformin 500mg twice daily and lisinopril 10mg once daily. "
    "Chest X-ray shows no evidence of pneumonia. "
    "Denies fever, cough, or recent travel."
)
doc = TextDocument(text=note_text)
print("Note:")
print(note_text)

In [ ]:
# --- Step 1: split into sentences (negation operates per-sentence) ---
sent_tok = SentenceTokenizer(output_label="SENTENCE")
sentences = sent_tok.run([doc.raw_segment])
for i, s in enumerate(sentences, 1):
    print(f"{i}. {s.text}")

In [ ]:
# --- Step 2: NER for diseases, symptoms, drugs ---
ner_rules = [
    # Diseases
    RegexpMatcherRule(label="DISEASE", regexp=r"\b(?:type [12] )?diabetes\b"),
    RegexpMatcherRule(label="DISEASE", regexp=r"\bhypertension\b"),
    RegexpMatcherRule(label="DISEASE", regexp=r"\bpneumonia\b"),
    # Symptoms
    RegexpMatcherRule(label="SYMPTOM", regexp=r"\bchest pain\b"),
    RegexpMatcherRule(label="SYMPTOM", regexp=r"\bshortness of breath\b"),
    RegexpMatcherRule(label="SYMPTOM", regexp=r"\bfever\b"),
    RegexpMatcherRule(label="SYMPTOM", regexp=r"\bcough\b"),
    # Drugs
    RegexpMatcherRule(label="DRUG",    regexp=r"\bmetformin\b"),
    RegexpMatcherRule(label="DRUG",    regexp=r"\blisinopril\b"),
]
matcher  = RegexpMatcher(rules=ner_rules)
entities = matcher.run([doc.raw_segment])

print(f"Found {len(entities)} entities:")
for e in entities:
    print(f"  {e.label:>8} | {e.text}")

In [ ]:
# --- Step 3: negation detection on the sentences ---
neg_rules = [
    NegationDetectorRule(regexp=r"\bno evidence of\b"),
    NegationDetectorRule(regexp=r"\bdenies\b"),
    NegationDetectorRule(regexp=r"\bno\s"),
    NegationDetectorRule(regexp=r"\bnegative for\b"),
    NegationDetectorRule(regexp=r"\brule[ds]? out\b"),
]
neg_detector = NegationDetector(output_label="is_negated", rules=neg_rules)
neg_detector.run(sentences)

# --- Step 4: for each entity, find its sentence and propagate negation ---
def sentence_of(entity, sentences):
    e0 = entity.spans[0].start
    eN = entity.spans[-1].end
    for s in sentences:
        if s.spans[0].start <= e0 and eN <= s.spans[-1].end:
            return s
    return None

rows = []
for e in entities:
    s = sentence_of(e, sentences)
    is_neg_attrs = s.attrs.get(label="is_negated") if s else []
    is_negated   = is_neg_attrs[0].value if is_neg_attrs else False
    rows.append({"label": e.label, "text": e.text,
                 "negated": is_negated,
                 "sentence": s.text if s else None})

clinical_df = pd.DataFrame(rows)
clinical_df

In [ ]:
# Quick sanity check: positive vs negated findings per category
clinical_df.groupby(["label", "negated"]).size().unstack(fill_value=0)

**Takeaway.**

The pipeline above is the **minimum viable clinical NLP**: tokenize sentences → match entities → detect negation → propagate to entities. In a production system, you'd add:
- **Section detection** (separate History, Findings, Impression, Plan — drugs in History mean past meds, in Plan mean prescribed now)
- **Uncertainty / hedging** (`may have`, `cannot exclude`, `consistent with`)
- **Coreference & temporal expressions** (`last week`, `since admission`)
- **Ontology linking** (map `metformin` → RxNorm CUI; `pneumonia` → SNOMED 233604007)

For deep-learning-based NER, the standard alternatives are:
- **`scispaCy`** — biomedical spaCy models (`en_core_sci_md`, `en_ner_bc5cdr_md` for chemicals/diseases)
- **Hugging Face transformers** — clinical BERT variants (`emilyalsentzer/Bio_ClinicalBERT`, `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`)

Use `medkit` to **orchestrate** these and keep the pipeline reproducible.


## 1.5 Cross-modality summary

| | X-ray | MRI | WSI | EHR |
|---|---|---|---|---|
| **File size** | KB–MB per file | 10–500 MB per volume | 0.5–4 GB per slide | KB of text |
| **Dimensionality** | 2D image + header | 3D or 4D volume | 2D, gigapixel, pyramidal | 1D token stream |
| **Key library** | `pydicom` | `nibabel` | `openslide-python` | `medkit` |
| **Geometry library** | `SimpleITK` | `nibabel` / `SimpleITK` / `MONAI` | OpenSlide MPP property | n/a |
| **Loads into RAM?** | Yes, easily | Yes, with care | Never — tile-based | Yes |
| **Key metadata** | DICOM header tags | NIfTI affine + zooms | MPP, objective, vendor | sentence/section structure |
| **Main pitfall** | Compressed transfer syntaxes (JPEG2000, RLE) need extra decoders | Orientation/affine — never stack raw pixels across volumes | Memory — always read tiles, not the whole slide | Negation, hedging, coreference |

**Practical workflow patterns.**
- A radiology report-pair pipeline → `pydicom` (images) + `medkit` (matched reports).
- A multi-modal MRI tumour-segmentation training set → `nibabel` for I/O + `MONAI` for transforms and dataloaders.
- A pathology-slide classifier → `openslide-python` for tile streaming + a CNN that ingests 256×256 patches.
- A cohort-selection script that picks "all CT chests, age 60–80, acquired in 2024" → `pydicom` with `stop_before_pixels=True`, pandas filter.


---
# Task 2 — DICOM Metadata Explorer

**Goal (strictly per the assignment).** Load all DICOM files from a folder; extract **modality**, **patient age**, **acquisition date**, **body part examined**; store the results in a `pandas` DataFrame; generate summary statistics.

**Dataset.** **SIIM_SMALL** — 250 real chest-X-ray DICOMs from the SIIM-ACR Pneumothorax Segmentation challenge (~30 MB), downloaded automatically by `fastai`'s data loader. No Kaggle credentials needed. Alternative paths (full Kaggle SIIM-FISABIO-RSNA, synthetic offline fallback) are at the end.


## 2.1 Setup

In [ ]:
import os, glob
import pydicom
import pandas as pd
import matplotlib.pyplot as plt

## 2.2 Download the dataset

In [ ]:
from fastai.data.external import untar_data, URLs

DATA_PATH = untar_data(URLs.SIIM_SMALL)
FOLDER    = str(DATA_PATH / "train")
print("Downloaded to:", DATA_PATH)
print("Will explore: ", FOLDER)

## 2.3 Helper — parse the DICOM `PatientAge` field

DICOM specifies age as 3 digits + a unit letter (`Y`/`M`/`W`/`D`), e.g. `045Y`. But many real datasets — SIIM_SMALL included — store it as a plain integer string (`"21"`). The parser handles both.

In [ ]:
def parse_age(age_str):
    """Convert a DICOM PatientAge string -> age in years (float), or None."""
    if not age_str or not isinstance(age_str, str):
        return None
    age_str = age_str.strip()
    if not age_str:
        return None
    # Spec format: digits + unit letter
    if age_str[-1].upper() in "YMWD":
        try:
            v = int(age_str[:-1])
        except ValueError:
            return None
        return {"Y": v, "M": v/12, "W": v*7/365, "D": v/365}[age_str[-1].upper()]
    # Plain integer (common in SIIM_SMALL)
    try:
        return float(age_str)
    except ValueError:
        return None

for s in ["045Y", "018M", "009D", "21", "", None, "bad"]:
    print(f"{repr(s):>8} -> {parse_age(s)}")

## 2.4 Core function — extract the four required fields

Design choices:
- **`stop_before_pixels=True`** — we only need headers; skipping pixel decode makes this ~100× faster on large folders.
- **`getattr(ds, "Tag", None)`** — real-world DICOMs have inconsistent tag presence; a missing tag becomes NaN, not a crash.
- **`recursive=True`** — SIIM_SMALL nests files in `train/Pneumothorax/` and `train/No Pneumothorax/`.
- **Errors are recorded, not raised** — one corrupted file shouldn't abort the batch.

In [ ]:
def explore_dicom_folder(folder, recursive=True):
    """
    Walk `folder` for DICOM files and return a DataFrame with the four
    fields required by the assignment, plus a few useful extras.
    """
    pattern = os.path.join(folder, "**", "*.dcm") if recursive else os.path.join(folder, "*.dcm")
    paths   = sorted(glob.glob(pattern, recursive=recursive))

    rows = []
    for path in paths:
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        except Exception as e:
            rows.append({"file": os.path.basename(path), "error": str(e)})
            continue
        if not hasattr(ds, "Modality"):
            continue

        rows.append({
            "file":                os.path.basename(path),
            # --- the four required fields ---
            "modality":            getattr(ds, "Modality", None),
            "patient_age":         parse_age(getattr(ds, "PatientAge", None)),
            "acquisition_date":    getattr(ds, "AcquisitionDate",
                                   getattr(ds, "StudyDate", None)),
            "body_part_examined":  getattr(ds, "BodyPartExamined", None),
            # --- handy extras ---
            "patient_id":          getattr(ds, "PatientID", None),
            "patient_sex":         getattr(ds, "PatientSex", None),
            "view_position":       getattr(ds, "ViewPosition", None),
            "manufacturer":        getattr(ds, "Manufacturer", None),
        })

    df = pd.DataFrame(rows)
    df["acquisition_date"] = pd.to_datetime(
        df["acquisition_date"], format="%Y%m%d", errors="coerce"
    )
    return df

## 2.5 Run the explorer

In [ ]:
df = explore_dicom_folder(FOLDER, recursive=True)
print(f"Read {len(df)} DICOM file(s) from {FOLDER!r}")
df.head(10)

## 2.6 Summary statistics

In [ ]:
print("=" * 50)
print("Modality distribution")
print("=" * 50)
print(df["modality"].value_counts(dropna=False).to_string())

print("\n" + "=" * 50)
print("Body part examined")
print("=" * 50)
print(df["body_part_examined"].value_counts(dropna=False).to_string())

print("\n" + "=" * 50)
print("Patient age (years) — descriptive statistics")
print("=" * 50)
print(df["patient_age"].describe().round(1).to_string())

print("\n" + "=" * 50)
print("Acquisition date range")
print("=" * 50)
print(f"earliest : {df['acquisition_date'].min()}")
print(f"latest   : {df['acquisition_date'].max()}")
print(f"unique dates: {df['acquisition_date'].nunique()}")

print("\n" + "=" * 50)
print("Patient sex")
print("=" * 50)
print(df["patient_sex"].value_counts(dropna=False).to_string())

print("\n" + "=" * 50)
print("Missing values per column")
print("=" * 50)
print(df.isna().sum().to_string())

## 2.7 Visual summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["modality"].value_counts().plot.bar(
    ax=axes[0], color="steelblue", edgecolor="black"
)
axes[0].set_title("DICOM files per modality")
axes[0].set_xlabel("Modality")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

df["body_part_examined"].value_counts(dropna=False).plot.bar(
    ax=axes[1], color="darkorange", edgecolor="black"
)
axes[1].set_title("Files per body part")
axes[1].set_xlabel("Body part")
axes[1].tick_params(axis="x", rotation=20)

df["patient_age"].dropna().plot.hist(
    ax=axes[2], bins=20, color="seagreen", edgecolor="black"
)
axes[2].set_title("Patient age distribution")
axes[2].set_xlabel("Age (years)")

plt.tight_layout()
plt.show()

---
# Appendix — Alternative dataset sources for Task 2

## A. Kaggle API — full SIIM-FISABIO-RSNA COVID-19 Detection set

The dataset linked in the original assignment (~120 GB unzipped, too big for a class demo, but documented here for completeness).

```python
# Step 1 — kaggle.com > Settings > API > Create New API Token, then upload kaggle.json
from google.colab import files
files.upload()                          # pick kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Step 2 — accept the competition rules on the Kaggle page (CLI returns 403 otherwise),
# then download:
!pip install kaggle --quiet
!kaggle competitions download -c siim-covid19-detection -p siim_full
!unzip -q siim_full/siim-covid19-detection.zip -d siim_full

# Step 3 — point the explorer at the unzipped folder, nothing else changes:
FOLDER = "siim_full/train"
df     = explore_dicom_folder(FOLDER, recursive=True)
```

## B. Offline fallback — synthetic DICOMs

For air-gapped environments or firewalls. Generates 8 small DICOMs with realistic tag values.

In [ ]:
# Run this cell only if you cannot download SIIM_SMALL
import numpy as np
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import generate_uid, ExplicitVRLittleEndian

os.makedirs("sample_dicoms", exist_ok=True)

patients = [
    {"id":"P001","age":"045Y","sex":"M","modality":"CR","body_part":"CHEST",  "date":"20240312"},
    {"id":"P002","age":"067Y","sex":"F","modality":"CT","body_part":"CHEST",  "date":"20240315"},
    {"id":"P003","age":"032Y","sex":"F","modality":"MR","body_part":"BRAIN",  "date":"20240401"},
    {"id":"P004","age":"058Y","sex":"M","modality":"CT","body_part":"ABDOMEN","date":"20240405"},
    {"id":"P005","age":"071Y","sex":"M","modality":"CR","body_part":"CHEST",  "date":"20240410"},
    {"id":"P006","age":"024Y","sex":"F","modality":"MR","body_part":"KNEE",   "date":"20240412"},
    {"id":"P007","age":"055Y","sex":"M","modality":"CR","body_part":"CHEST",  "date":"20240418"},
    {"id":"P008","age":"049Y","sex":"F","modality":"CT","body_part":"CHEST",  "date":"20240420"},
]
for p in patients:
    suid = generate_uid()
    fm = FileMetaDataset()
    fm.MediaStorageSOPClassUID    = "1.2.840.10008.5.1.4.1.1.2"
    fm.MediaStorageSOPInstanceUID = suid
    fm.TransferSyntaxUID          = ExplicitVRLittleEndian
    fname = f"sample_dicoms/{p['id']}.dcm"
    ds = FileDataset(fname, {}, file_meta=fm, preamble=b"\0"*128)
    ds.PatientID, ds.PatientName     = p["id"], f"Anonymous^{p['id']}"
    ds.PatientAge, ds.PatientSex     = p["age"], p["sex"]
    ds.Modality, ds.BodyPartExamined = p["modality"], p["body_part"]
    ds.StudyDate, ds.AcquisitionDate = p["date"], p["date"]
    ds.Manufacturer                  = "DemoCorp"
    ds.StudyInstanceUID  = generate_uid()
    ds.SeriesInstanceUID = generate_uid()
    ds.SOPInstanceUID    = suid
    ds.SOPClassUID       = fm.MediaStorageSOPClassUID
    arr = (np.random.rand(32, 32) * 255).astype(np.uint8)
    ds.Rows, ds.Columns       = arr.shape
    ds.BitsAllocated, ds.BitsStored, ds.HighBit = 8, 8, 7
    ds.PixelRepresentation, ds.SamplesPerPixel  = 0, 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.PixelData = arr.tobytes()
    ds.save_as(fname, enforce_file_format=True)

# Realistic touch: drop a tag from one file so we have a NaN to handle
ds = pydicom.dcmread("sample_dicoms/P005.dcm")
del ds.BodyPartExamined
ds.save_as("sample_dicoms/P005.dcm")

# Then point the explorer at it:
df_synth = explore_dicom_folder("sample_dicoms")
df_synth